In [6]:
import json
import google.auth
from google.cloud import discoveryengine_v1alpha as discoveryengine_v1
from google.api_core.client_options import ClientOptions
from dotenv import load_dotenv
import os

load_dotenv()

True

In [7]:
datastore_id = os.getenv("DATASTORE_ID")
location = "us"
GOOGLE_CLOUD_PROJECT = os.getenv("GOOGLE_CLOUD_PROJECT")

In [8]:
SEARCH_APP_LOCATION = location
client_options = ClientOptions(api_endpoint=f"{SEARCH_APP_LOCATION}-discoveryengine.googleapis.com")
doc_client = discoveryengine_v1.DocumentServiceClient(client_options=client_options)

In [9]:
request = discoveryengine_v1.ListDocumentsRequest(
    parent=f"projects/{GOOGLE_CLOUD_PROJECT}/locations/{location}/collections/default_collection/dataStores/{datastore_id}/branches/default_branch",
    page_size=1000
)

documents = list(doc_client.list_documents(request=request))
print(f"Fetched {len(documents)} documents.")

Fetched 16 documents.


In [10]:
# List all docs: doc_id | source_id | title
# discoveryengine_v1alpha is a proto-plus client.
# struct_data is already a plain Python dict — do NOT use MessageToDict on it.

print(f"Total documents: {len(documents)}\n")
print(f"{'doc_id':<45} {'source_id':<15} title")
print("-" * 110)

for doc in documents:
    doc_id = doc.name.split("/")[-1]
    meta = {**(doc.struct_data or {}), **(doc.derived_struct_data or {})}
    source_id = meta.get("source_id") or meta.get("sourceId") or ""
    title     = meta.get("title")     or meta.get("name")      or ""
    print(f"{doc_id:<45} {source_id:<15} {title}")

Total documents: 16

doc_id                                        source_id       title
--------------------------------------------------------------------------------------------------------------
DOC_ID_305                                    DOC_ID_305      DP PC Study Updates: DS Inversion and FBDS Mixing
DOC_ID_306                                    DOC_ID_306      Automation Improvements for Adams 0.5mL Fill
DOC_ID_307                                    DOC_ID_307      DP PC dashboard (5/28)
DOC_ID_308                                    DOC_ID_308      [COMPOUND_8] Kick-off
DOC_ID_309                                    DOC_ID_309      Ixo Vec Gap Assessment Technical Studies Quality Review
DOC_ID_310                                    DOC_ID_310      Analytical Deep Dive
DOC_ID_311                                    DOC_ID_311      Meeting Summary – Weekly: [COMPOUND_8] DP PC Team
DOC_ID_312                                    DOC_ID_312      Drug Product Process Characterization

In [11]:
# Find a specific document by source_id
TARGET_SOURCE_ID = "DOC_ID_506"

sample_doc = None
for doc in documents:
    meta = {**(doc.struct_data or {}), **(doc.derived_struct_data or {})}
    if meta.get("source_id") == TARGET_SOURCE_ID or meta.get("sourceId") == TARGET_SOURCE_ID:
        sample_doc = doc
        break

if sample_doc:
    print(f"Found: {sample_doc.name}")
else:
    print(f"No document with source_id={TARGET_SOURCE_ID!r} found.")

No document with source_id='DOC_ID_506' found.


In [ ]:
chunk_client = discoveryengine_v1.ChunkServiceClient(client_options=client_options)

In [ ]:
chunks = chunk_client.list_chunks(parent=sample_doc.name)

for chunk in chunks:
    print(f"  chunk id      : {chunk.id}")
    print(f"  content[:120] : {chunk.content[:120]}")
    print(f"  doc uri       : {chunk.document_metadata.uri}")
    print(f"  doc title     : {chunk.document_metadata.title}")
    print()
    break  # remove to see all chunks